In [1]:
```python
# ============================================================
# NLP + K-MEANS CLUSTERING FOR us.csv
# Complete Jupyter Notebook Code
# ============================================================


# ============================================================
# CELL 1: INSTALL / IMPORT LIBRARIES
# ============================================================

# If required, uncomment and run:
# !pip install pandas numpy matplotlib scikit-learn nltk

import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt

import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import TruncatedSVD


# Download NLTK resources
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")


print("Libraries imported successfully.")


# ============================================================
# CELL 2: LOAD DATASET
# ============================================================

file_name = "us.csv"

df = pd.read_csv(file_name)

print("Dataset loaded successfully.")
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

display(df.head())


# ============================================================
# CELL 3: BASIC DATASET INFORMATION
# ============================================================

print("=" * 60)
print("DATASET INFORMATION")
print("=" * 60)

print("\nShape:")
print(df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:")
print(df.duplicated().sum())


# ============================================================
# CELL 4: REMOVE DUPLICATES
# ============================================================

before = len(df)

df = df.drop_duplicates().reset_index(drop=True)

after = len(df)

print("Duplicate rows removed:", before - after)
print("New dataset size:", df.shape)


# ============================================================
# CELL 5: IDENTIFY TEXT COLUMNS
# ============================================================

text_columns = df.select_dtypes(
    include=["object", "string"]
).columns.tolist()

print("=" * 60)
print("TEXT COLUMNS")
print("=" * 60)

print(text_columns)


# Check whether text columns exist
if len(text_columns) == 0:
    raise ValueError(
        "No text columns were found in us.csv. "
        "K-Means NLP requires at least one text column."
    )


# ============================================================
# CELL 6: HANDLE MISSING VALUES
# ============================================================

for column in text_columns:
    df[column] = df[column].fillna("")

print("Missing values in text columns replaced with empty strings.")


# ============================================================
# CELL 7: COMBINE ALL TEXT COLUMNS
# ============================================================

df["combined_text"] = df[text_columns].astype(str).agg(
    " ".join,
    axis=1
)

print("=" * 60)
print("COMBINED TEXT")
print("=" * 60)

display(df[["combined_text"]].head())


# ============================================================
# CELL 8: NLP TEXT CLEANING
# ============================================================

stop_words = set(stopwords.words("english"))

lemmatizer = WordNetLemmatizer()


def clean_text(text):
    """
    Clean and preprocess text.
    """

    # Convert to string
    text = str(text)

    # Convert to lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(
        r"http\S+|www\S+|https\S+",
        " ",
        text
    )

    # Remove email addresses
    text = re.sub(
        r"\S+@\S+",
        " ",
        text
    )

    # Remove HTML tags
    text = re.sub(
        r"<.*?>",
        " ",
        text
    )

    # Remove numbers
    text = re.sub(
        r"\d+",
        " ",
        text
    )

    # Remove punctuation and special characters
    text = re.sub(
        r"[^a-zA-Z\s]",
        " ",
        text
    )

    # Remove extra spaces
    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    # Tokenize
    words = text.split()

    # Remove stopwords and short words
    words = [
        word
        for word in words
        if word not in stop_words
        and len(word) > 2
    ]

    # Lemmatization
    words = [
        lemmatizer.lemmatize(word)
        for word in words
    ]

    # Return cleaned text
    return " ".join(words)


# Apply cleaning
df["clean_text"] = df["combined_text"].apply(clean_text)


print("=" * 60)
print("CLEANED TEXT")
print("=" * 60)

display(
    df[["combined_text", "clean_text"]].head(10)
)


# ============================================================
# CELL 9: REMOVE EMPTY TEXT RECORDS
# ============================================================

before = len(df)

df = df[
    df["clean_text"].str.strip() != ""
].reset_index(drop=True)

after = len(df)

print("Rows removed because of empty text:", before - after)
print("Rows remaining:", after)


# ============================================================
# CELL 10: TF-IDF VECTORIZATION
# ============================================================

vectorizer = TfidfVectorizer(
    max_features=5000,
    min_df=2,
    max_df=0.95,
    ngram_range=(1, 2),
    sublinear_tf=True
)

X = vectorizer.fit_transform(
    df["clean_text"]
)

print("=" * 60)
print("TF-IDF INFORMATION")
print("=" * 60)

print("TF-IDF matrix shape:", X.shape)
print("Number of documents:", X.shape[0])
print("Number of features:", X.shape[1])


# ============================================================
# CELL 11: DISPLAY TF-IDF FEATURES
# ============================================================

features = vectorizer.get_feature_names_out()

print("First 50 TF-IDF features:")
print(features[:50])


# ============================================================
# CELL 12: ELBOW METHOD
# ============================================================

inertia = []

k_values = range(2, 11)

for k in k_values:

    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    model.fit(X)

    inertia.append(model.inertia_)


# Plot Elbow Method

plt.figure(figsize=(10, 6))

plt.plot(
    k_values,
    inertia,
    marker="o"
)

plt.xlabel("Number of Clusters (K)")
plt.ylabel("Inertia")
plt.title("Elbow Method")

plt.xticks(k_values)
plt.grid(True)

plt.show()


# ============================================================
# CELL 13: SILHOUETTE SCORE
# ============================================================

silhouette_scores = []

for k in range(2, 11):

    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    labels = model.fit_predict(X)

    score = silhouette_score(
        X,
        labels
    )

    silhouette_scores.append(score)

    print(
        f"K = {k} | "
        f"Silhouette Score = {score:.4f}"
    )


# Plot silhouette scores

plt.figure(figsize=(10, 6))

plt.plot(
    range(2, 11),
    silhouette_scores,
    marker="o"
)

plt.xlabel("Number of Clusters (K)")
plt.ylabel("Silhouette Score")
plt.title("Silhouette Score")

plt.xticks(range(2, 11))
plt.grid(True)

plt.show()


# ============================================================
# CELL 14: SELECT BEST K
# ============================================================

best_k = list(range(2, 11))[
    np.argmax(silhouette_scores)
]

best_score = max(silhouette_scores)

print("=" * 60)
print("BEST NUMBER OF CLUSTERS")
print("=" * 60)

print("Best K:", best_k)
print("Best Silhouette Score:", round(best_score, 4))


# ============================================================
# CELL 15: TRAIN FINAL K-MEANS MODEL
# ============================================================

kmeans = KMeans(
    n_clusters=best_k,
    random_state=42,
    n_init=10
)

df["cluster"] = kmeans.fit_predict(X)

print("K-Means model trained successfully.")


# ============================================================
# CELL 16: CLUSTER COUNTS
# ============================================================

cluster_counts = (
    df["cluster"]
    .value_counts()
    .sort_index()
)

print("=" * 60)
print("NUMBER OF RECORDS PER CLUSTER")
print("=" * 60)

display(
    cluster_counts.to_frame(
        name="Number of Records"
    )
)


# ============================================================
# CELL 17: CLUSTER SIZE VISUALIZATION
# ============================================================

plt.figure(figsize=(10, 6))

plt.bar(
    cluster_counts.index.astype(str),
    cluster_counts.values
)

plt.xlabel("Cluster")
plt.ylabel("Number of Records")
plt.title("Number of Records in Each Cluster")

plt.show()


# ============================================================
# CELL 18: FIND TOP WORDS FOR EACH CLUSTER
# ============================================================

terms = vectorizer.get_feature_names_out()

cluster_centers = kmeans.cluster_centers_


print("=" * 60)
print("TOP WORDS IN EACH CLUSTER")
print("=" * 60)


for cluster_number in range(best_k):

    # Get feature importance for this cluster
    center = cluster_centers[cluster_number]

    # Sort features by importance
    top_indices = center.argsort()[-20:][::-1]

    top_words = [
        terms[index]
        for index in top_indices
    ]

    print()
    print(f"CLUSTER {cluster_number}")
    print("-" * 50)

    print(
        ", ".join(top_words)
    )


# ============================================================
# CELL 19: CREATE CLUSTER SUMMARY
# ============================================================

cluster_summary = []

for cluster_number in range(best_k):

    center = cluster_centers[cluster_number]

    top_indices = center.argsort()[-15:][::-1]

    top_words = [
        terms[index]
        for index in top_indices
    ]

    count = (
        df["cluster"] == cluster_number
    ).sum()

    cluster_summary.append({
        "Cluster": cluster_number,
        "Number_of_Records": count,
        "Top_Words": ", ".join(top_words)
    })


cluster_summary_df = pd.DataFrame(
    cluster_summary
)


print("=" * 60)
print("CLUSTER SUMMARY")
print("=" * 60)

display(cluster_summary_df)


# ============================================================
# CELL 20: DISPLAY EXAMPLES FROM EACH CLUSTER
# ============================================================

for cluster_number in range(best_k):

    print()
    print("=" * 70)
    print(f"CLUSTER {cluster_number}")
    print("=" * 70)

    cluster_data = df[
        df["cluster"] == cluster_number
    ]

    # Display up to 10 records
    display(
        cluster_data[
            text_columns + ["cluster"]
        ].head(10)
    )


# ============================================================
# CELL 21: REDUCE TF-IDF TO 2 DIMENSIONS
# ============================================================

svd = TruncatedSVD(
    n_components=2,
    random_state=42
)

X_2d = svd.fit_transform(X)

print("2D representation created.")
print("Shape:", X_2d.shape)


# ============================================================
# CELL 22: VISUALIZE K-MEANS CLUSTERS
# ============================================================

plt.figure(figsize=(12, 8))

scatter = plt.scatter(
    X_2d[:, 0],
    X_2d[:, 1],
    c=df["cluster"],
    alpha=0.6
)

plt.xlabel("SVD Component 1")
plt.ylabel("SVD Component 2")

plt.title(
    "K-Means Clustering of us.csv"
)

plt.colorbar(
    scatter,
    label="Cluster"
)

plt.grid(True)

plt.show()


# ============================================================
# CELL 23: ADD 2D COORDINATES TO DATASET
# ============================================================

df["SVD_1"] = X_2d[:, 0]
df["SVD_2"] = X_2d[:, 1]


# ============================================================
# CELL 24: SHOW FINAL DATASET
# ============================================================

print("=" * 60)
print("FINAL DATASET")
print("=" * 60)

display(df.head())


# ============================================================
# CELL 25: SAVE CLUSTERED DATASET
# ============================================================

output_file = "us_kmeans_clusters.csv"

# Remove temporary NLP columns
output_df = df.drop(
    columns=[
        "combined_text",
        "clean_text"
    ],
    errors="ignore"
)

output_df.to_csv(
    output_file,
    index=False
)

print()
print("Clustering results saved successfully.")
print("File:", output_file)


# ============================================================
# CELL 26: SAVE CLUSTER SUMMARY
# ============================================================

summary_file = "us_cluster_summary.csv"

cluster_summary_df.to_csv(
    summary_file,
    index=False
)

print(
    "Cluster summary saved as:",
    summary_file
)


# ============================================================
# CELL 27: FINAL REPORT
# ============================================================

print()
print("=" * 70)
print("FINAL K-MEANS NLP REPORT")
print("=" * 70)

print(
    f"Original number of rows: {before}"
)

print(
    f"Final number of rows: {len(df)}"
)

print(
    f"Number of text columns: {len(text_columns)}"
)

print(
    f"Number of TF-IDF features: {X.shape[1]}"
)

print(
    f"Optimal number of clusters: {best_k}"
)

print(
    f"Silhouette score: {best_score:.4f}"
)

print()
print("Cluster sizes:")

for cluster_number, count in cluster_counts.items():

    print(
        f"Cluster {cluster_number}: "
        f"{count} records"
    )

print()
print("Output files:")
print("1. us_kmeans_clusters.csv")
print("2. us_cluster_summary.csv")

print()
print("Analysis completed successfully!")
```

### What this complete notebook does

**NLP:** cleans the text, removes stopwords, removes punctuation/numbers, and lemmatizes words.

**TF-IDF:** converts the text into numerical features suitable for machine learning.

**K-Means:** automatically tests `K = 2` through `K = 10`.

**Model selection:** uses both the **Elbow Method** and **Silhouette Score** to help determine the appropriate number of clusters.

**Cluster interpretation:** extracts the top words associated with every cluster.

**Visualization:** produces cluster-size charts and a 2D visualization of the K-Means results.

**Output:** creates:

* `us_kmeans_clusters.csv` — your original data with cluster assignments.
* `us_cluster_summary.csv` — cluster sizes and their most important words.

If your `us.csv` has a **specific structure (for example, columns such as `title`, `description`, `review`, `category`, etc.)**, uploading the file would let me tailor the code precisely to those columns rather than automatically combining every text column.


SyntaxError: invalid decimal literal (2974142322.py, line 720)